# Validate crops

This notebook performs validation on the preprocessing steps for images to ensure that the resulting crops are the same as what is uploaded to HuggingFace.

In particular, this uses the `get_drop` function from https://huggingface.co/Addax-Data-Science/NZI-ADS-v1/blob/main/inference.py

## Setup

In [2]:
import requests
import base64
import json
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt
import io
import numpy as np
from io import BytesIO
import math

## HuggingFace crops

The only changes from `inference.py` on HuggingFace are the function names:
- `get_drop` -> `get_addax_crop`
- `_pad_crop` -> `_pad_addax_crop`

In [8]:
def get_addax_crop(
    image: Image.Image, bbox: tuple[float, float, float, float]
) -> Image.Image:
    """
    Crop image using model-specific preprocessing.
    This cropping method was developed by Dan Morris for MegaDetector and is
    designed to:
    1. Square the bounding box (max of width/height)
    2. Add padding to prevent over-enlargement of small animals
    3. Center the detection within the crop
    4. Pad with black (0) to maintain square aspect ratio
    Args:
        image: PIL Image (full resolution)
        bbox: Normalized bounding box (x, y, width, height) in range [0.0, 1.0]
    Returns:
       Cropped and padded PIL Image ready for classification
    Raises:
        ValueError: If bbox is invalid (zero size)
    """
    img_w, img_h = image.size

    # Denormalize bbox coordinates
    xmin = int(bbox[0] * img_w)
    ymin = int(bbox[1] * img_h)
    box_w = int(bbox[2] * img_w)
    box_h = int(bbox[3] * img_h)

    # Square the box (use max dimension)
    box_size = max(box_w, box_h)

    # Add padding (prevents over-enlargement of small animals)
    box_size = pad_addax_crop(box_size)

    # Center the detection within the squared crop
    xmin = max(0, min(xmin - int((box_size - box_w) / 2), img_w - box_w))
    ymin = max(0, min(ymin - int((box_size - box_h) / 2), img_h - box_h))

    # Clip to image boundaries
    box_w = min(img_w, box_size)
    box_h = min(img_h, box_size)

    if box_w == 0 or box_h == 0:
        raise ValueError(f"Invalid bbox size: {box_w}x{box_h}")

    # Crop and pad to square
    crop = image.crop(box=[xmin, ymin, xmin + box_w, ymin + box_h])
    crop = ImageOps.pad(crop, size=(box_size, box_size), color=0)

    return crop

def pad_addax_crop(box_size: int) -> int:
    """
    Calculate padded crop size to prevent over-enlargement of small animals.
    YOLOv8 expects 224x224 input. This function ensures small detections aren't
    excessively upscaled while adding consistent padding to larger detections.
    Args:
        box_size: Original bounding box size (max of width/height)
    Returns:
        Padded box size
    """
    input_size_network = 224
    default_padding = 30

    if box_size >= input_size_network:
        # Large detection: add default padding
        return box_size + default_padding
    else:
        # Small detection: ensure minimum size without excessive enlargement
        diff_size = input_size_network - box_size
        if diff_size < default_padding:
            return box_size + default_padding
        else:
            return input_size_network

## Animl crops

Taken from the `serve.py` file of nzi-adsv1 in Animl-ml https://github.com/tnc-ca-geo/animl-ml/pull/148/changes#diff-9b13620e8bb63cfb6f731b643c7e6ad9e932a1b4c25429cc60f16f11d77200fd

In [9]:
def get_animl_crop(image: Image.Image, bbox: list[float]) -> Image.Image:
    """
    Crop image using Dan Morris's MegaDetector preprocessing method.
    
    Args:
        image: PIL Image (full resolution)
        bbox: Normalized bounding box [x, y, width, height] in range [0.0, 1.0]
    
    Returns:
        Cropped and padded PIL Image ready for classification
    """
    img_w, img_h = image.size
    
    # Denormalize bbox coordinates
    xmin = int(bbox[0] * img_w)
    ymin = int(bbox[1] * img_h)
    box_w = int(bbox[2] * img_w)
    box_h = int(bbox[3] * img_h)
    
    # Square the box (use max dimension)
    box_size = max(box_w, box_h)
    
    # Add padding (prevents over-enlargement of small animals)
    input_size_network = 224
    default_padding = 30
    if box_size >= input_size_network:
        box_size = box_size + default_padding
    else:
        diff_size = input_size_network - box_size
        if diff_size < default_padding:
            box_size = box_size + default_padding
        else:
            box_size = input_size_network
    
    # Center the detection within the squared crop
    xmin = max(0, min(xmin - int((box_size - box_w) / 2), img_w - box_w))
    ymin = max(0, min(ymin - int((box_size - box_h) / 2), img_h - box_h))
    
    # Clip to image boundaries
    box_w = min(img_w, box_size)
    box_h = min(img_h, box_size)
    
    if box_w == 0 or box_h == 0:
        raise ValueError(f"Invalid bbox size: {box_w}x{box_h}")
    
    # Crop and pad to square
    crop = image.crop(box=[xmin, ymin, xmin + box_w, ymin + box_h])
    crop = ImageOps.pad(crop, size=(box_size, box_size), color=0)
    
    return crop

In [45]:
def compare_crops(crop1, crop2):
    """Compare two crops to see if they're identical"""

    # Check dimensions
    if crop1.size != crop2.size:
        print(f"❌ Different sizes: {crop1.size} vs {crop2.size}")
        return False

    # Convert to numpy arrays
    arr1 = np.array(crop1)
    arr2 = np.array(crop2)

    # Check if identical
    if np.array_equal(arr1, arr2):
        print(f"✅ Crops are IDENTICAL ({crop1.size})")
        return True

    # Calculate difference
    diff = np.abs(arr1.astype(float) - arr2.astype(float))
    max_diff = diff.max()
    mean_diff = diff.mean()
    num_different = np.count_nonzero(diff)
    total_pixels = arr1.size

    print(f"❌ Crops are DIFFERENT:")
    print(f"   Max pixel difference: {max_diff}")
    print(f"   Mean pixel difference: {mean_diff:.2f}")
    print(f"   Different pixels: {num_different:,} / {total_pixels:,} ({100*num_different/total_pixels:.2f}%)")

    # Show difference heatmap
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(crop1)
    axes[0].set_title("Crop 1")
    axes[0].axis('off')

    axes[1].imshow(crop2)
    axes[1].set_title("Crop 2")
    axes[1].axis('off')

    axes[2].imshow(diff.mean(axis=2), cmap='hot')
    axes[2].set_title("Difference (brighter = more different)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    return False


with open('deployment_002/image_recognition_file.json') as td:
    addax_data = json.load(td)

for img in addax_data['images']:
    image_path = f'deployment_002/{img['file']}'
    with open(image_path, "rb") as f:
        image_bytes = f.read()
        
    b64_image = base64.b64encode(image_bytes)
    image = Image.open(io.BytesIO(image_bytes))
    for det in img['detections']:
        bbox = det['bbox']
        
        addax_crop = get_addax_crop(image, bbox)
        animl_crop = get_animl_crop(image, bbox)
        
        compare_crops(addax_crop, animl_crop)

        print('')

✅ Crops are IDENTICAL ((286, 286))

✅ Crops are IDENTICAL ((557, 557))

✅ Crops are IDENTICAL ((314, 314))

✅ Crops are IDENTICAL ((440, 440))

✅ Crops are IDENTICAL ((495, 495))

✅ Crops are IDENTICAL ((353, 353))

✅ Crops are IDENTICAL ((1900, 1900))

✅ Crops are IDENTICAL ((983, 983))

✅ Crops are IDENTICAL ((1466, 1466))

✅ Crops are IDENTICAL ((457, 457))

✅ Crops are IDENTICAL ((315, 315))

✅ Crops are IDENTICAL ((318, 318))

✅ Crops are IDENTICAL ((663, 663))

✅ Crops are IDENTICAL ((884, 884))

✅ Crops are IDENTICAL ((489, 489))

✅ Crops are IDENTICAL ((893, 893))

✅ Crops are IDENTICAL ((663, 663))

✅ Crops are IDENTICAL ((613, 613))

✅ Crops are IDENTICAL ((614, 614))

✅ Crops are IDENTICAL ((409, 409))

✅ Crops are IDENTICAL ((612, 612))

✅ Crops are IDENTICAL ((439, 439))

✅ Crops are IDENTICAL ((402, 402))

✅ Crops are IDENTICAL ((585, 585))

✅ Crops are IDENTICAL ((485, 485))

✅ Crops are IDENTICAL ((224, 224))

✅ Crops are IDENTICAL ((347, 347))

✅ Crops are IDENTICAL ((

## Compare results

In [5]:
def load_image(image_path):
    with open(image_path, 'rb') as f:
        image_bytes = f.read()
    image_b64 = base64.b64encode(image_bytes)
    return Image.open(image_path), image_b64


with open('deployment_002/base_image_recognition_file.json') as td:
    ads_data = json.load(td)

ads_imgs_res = ads_data['images']
ads_class_mapping = ads_data['classification_categories']
for ads_img_res in ads_imgs_res:
    ads_img_name = ads_img_res['file']
    print(f'Evaluating {ads_img_name}..')

    our_predictions = []
    for det_i, ads_det in enumerate(ads_img_res['detections']):
        image, b64_ads_img = load_image(f'deployment_002/{ads_img_name}')
        
        ads_bbox = ads_det['bbox']
        ads_classifications = {}
        
        for c in ads_det['classifications']:
            ads_classifications[ads_class_mapping[c[0]]] = c[1]

        animl_response = requests.post(
            "http://localhost:8080/invocations",
            data=json.dumps({
                "bbox": ads_bbox,
                "image": b64_ads_img.decode("utf-8")
            })
        )

        our_classification_res = animl_response.json()

        print(f'Differences for detection {det_i} of {ads_img_name}')
        for our_class, our_conf in our_classification_res.items():
            rounded_our_conf = round(our_conf, 5)
            ads_conf = ads_classifications[our_class]
            if not math.isclose(rounded_our_conf, ads_conf):
                print(f'- {our_class}.  Expected: {ads_conf} but got: {rounded_our_conf}.  Model output: {our_conf}')
    print('')

Evaluating img_0001.jpg..
Differences for detection 0 of img_0001.jpg
Differences for detection 1 of img_0001.jpg

Evaluating img_0002.jpg..
Differences for detection 0 of img_0002.jpg
Differences for detection 1 of img_0002.jpg

Evaluating img_0003.jpg..
Differences for detection 0 of img_0003.jpg
Differences for detection 1 of img_0003.jpg

Evaluating img_0004.jpg..
Differences for detection 0 of img_0004.jpg

Evaluating img_0005.jpg..
Differences for detection 0 of img_0005.jpg

Evaluating img_0006.jpg..
Differences for detection 0 of img_0006.jpg

Evaluating img_0007.jpg..
Differences for detection 0 of img_0007.jpg

Evaluating img_0008.jpg..
Differences for detection 0 of img_0008.jpg

Evaluating img_0009.jpg..
Differences for detection 0 of img_0009.jpg
- lagomorph.  Expected: 0.04524 but got: 0.04523.  Model output: 0.04523495212197304

Evaluating img_0010.jpg..
Differences for detection 0 of img_0010.jpg
Differences for detection 1 of img_0010.jpg
- hedgehog.  Expected: 0.38641